# HW3: Atomic cashflows and the Visitor pattern

This assignment develops two skills:

1. **Model cashflow contracts.** Implement `ProductFixedAccruedCashflow` and `ProductOvernightIndexCashflow` using the provided `Product` and `ProductCashflow` bases. Understand their contractual fields, including dates, notional, currency, accrual conventions, index and spread. Each object is a building block for a larger product, such as a swap.
2. **Use the Visitor pattern.** Collect and display each product's attributes through a visitor. This demonstrates how to add an operation without changing the product classes.

Read [IV. Visitor Pattern](https://immense-chocolate-92e.notion.site/IV-Visitor-Pattern-2d4831d0b1d58191bc36ebae3d771d58); implement `accept()` and the display visitor to produce the tables below.

Complete the six `#TODO` blocks in `fixedincomelib/product/linear_products.py` and `fixedincomelib/product/product_display_visitor.py` before running the examples.

Use the provided `qfCreateProductxxx` APIs to create product objects and `qfDisplayProduct` to inspect their fields.

These objects also form a foundation for a later `value_engine`, which can combine their contract attributes with the necessary rates and market data to calculate cash amounts and present values. The engine reads the product objects directly; the display tables are for inspection.

In [1]:
import sys
import QuantLib as ql

from fixedincomelib import (
    qfCreateProductFixedAccruedCashflow,
    qfCreateProductOvernightIndexCashflow,
    qfDisplayProduct,
)
from fixedincomelib.product import (
    ProductCashflow,
    ProductFixedAccruedCashflow, ProductOvernightIndexCashflow,
)

print(f"Python {sys.version.split()[0]} | QuantLib {ql.__version__}")

Python 3.11.13 | QuantLib 1.41


## 1. Fixed-accrual cashflow

The API returns a `ProductFixedAccruedCashflow` object containing contract fields and an accrual year fraction. `qfDisplayProduct` displays its fields; the fraction is checked below. This class has no fixed-rate field.

In [2]:
fixed = qfCreateProductFixedAccruedCashflow(
    effective_date="2026-01-02",
    termination_date="2026-04-02",
    currency="USD",
    notional=1_000_000,
    accrual_basis="ACT/360",
    payment_date="2026-04-06",
    business_day_convention="F",
    holiday_convention="USGS",
)
assert isinstance(fixed, ProductCashflow)
assert isinstance(fixed, ProductFixedAccruedCashflow)
print("Created:", type(fixed).__name__)
qfDisplayProduct(fixed)

NotImplementedError: TODO 1: ProductFixedAccruedCashflow.__init__

In [ ]:
# 90 calendar days / 360. This number is a year fraction, not a cash amount.
assert fixed.accrued == 90 / 360
assert fixed.termination_date.ISO() == "2026-04-02"
assert fixed.payment_date.ISO() == "2026-04-06"
assert fixed.first_date == fixed.effective_date
assert fixed.last_date == fixed.payment_date
print(f"Accrual year fraction: {fixed.accrued:.6f}")
print("Contractual accrual end:", fixed.termination_date.ISO())
print("Payment date:", fixed.payment_date.ISO())

## 2. Overnight-index cashflow

The API returns a `ProductOvernightIndexCashflow` object. It resolves the tenor to a termination date, obtains currency from the index, and stores the contract fields.

`0.001` is 10 basis points as an annual decimal rate. Negative notional implies `SHORT`. Creation and display do not calculate an interest amount.

In [ ]:
overnight = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert isinstance(overnight, ProductCashflow)
assert isinstance(overnight, ProductOvernightIndexCashflow)
assert overnight.notional == -1_000_000
assert overnight.currency.value_str == "USD"
print("Created:", type(overnight).__name__)
qfDisplayProduct(overnight)

In [ ]:
# Date and tenor inputs resolve to the same termination date.
overnight_by_date = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="2026-04-02",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert overnight.termination_date == overnight_by_date.termination_date
assert overnight.payment_date == overnight_by_date.payment_date
print("Date and tenor inputs agree:", overnight.termination_date.ISO())

## 3. Currency and payment defaults

Supported indices are `SOFR-1B`, `FF-1B` and `SONIA-1B` (case-insensitive). Currency comes from the index, and an omitted payment date defaults to termination. Compounding labels are `simple`, `arithmetic` and `compound`.

In [ ]:
sonia = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SONIA-1B",
    notional=500_000,
    compounding_method="arithmetic",
)
assert sonia.currency.value_str == "GBP"
assert sonia.payment_date == sonia.termination_date
qfDisplayProduct(sonia)